# Índice FIBRAS de AMEFIBRA

Notebook para extraer la tabla pública del Índice FIBRAS. La página carga los datos dentro de un `iframe` mediante JavaScript y WebSocket, por lo que se utiliza Playwright con Chromium.

> La información se ofrece únicamente para consulta y análisis. AMEFIBRA indica que los datos tienen aproximadamente 20 minutos de retraso y no deben usarse como base única para decisiones de inversión.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

try:
    # VS Code inyecta esta variable con la ruta absoluta del propio notebook,
    # así que la raíz del proyecto queda anclada a dónde vive el archivo .ipynb,
    # sin importar cuál sea el directorio de trabajo con el que arrancó el kernel
    # (que puede no ser la raíz del proyecto, según la configuración del editor).
    RAIZ_PROYECTO = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    RAIZ_PROYECTO = Path.cwd()
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

from modules.presentacion import (
    ejecutar_extraccion_indice,
    exportar_csv_excel,
    exportar_ficha_a_pdf,
    exportar_xlsx,
    mostrar_emisoras,
    mostrar_ficha_ejemplo_cliente,
    mostrar_ficha_rendimiento,
    probar_historial_dividendos,
    seleccionar_anio_interactivo,
    seleccionar_ticker_interactivo,
)
from modules.procesamiento import obtener_anios_disponibles

In [2]:
CARPETA_SALIDA = Path.cwd() / "output"
CARPETA_FICHAS_PDF = CARPETA_SALIDA / "fichas"

# Parámetros editables del notebook
HEADLESS = True
TIMEOUT_DATOS_MS = 30000
EXPORTAR_CSV_ANALITICO = True
EXPORTAR_CSV_EXCEL = False
EXPORTAR_XLSX = False
RUTA_CSV_EXCEL = Path.cwd() / "indice_fibras.csv"
RUTA_XLSX = Path.cwd() / "indice_fibras.xlsx"

## Ejecutar extracción
Consultar solo los lunes temprano para hacer un análisis rápido de las FIBRAS y para saber si hubo altas y bajas de emisoras.

In [3]:
df = ejecutar_extraccion_indice(HEADLESS, TIMEOUT_DATOS_MS, CARPETA_SALIDA, EXPORTAR_CSV_ANALITICO)

Consultando https://amefibra.com/el-mercado/indice-fibras/ ...


Índice FIBRAS - 2026-08-25 23:28 (dato con ~20 min de retraso)


Aviso: se agotó el tiempo esperando datos en vivo; se usará lo cargado.


,Emisora,Cotización,Var.,Var. %,Apertura,Máx. día,Min. día,Promedio,Operaciones,Volumen,Importe,Máx. 52 s.,Min. 52 s.
0,DANHOS13,28.77,0.02,0.07%,28.54,28.60,28.82,28.25,1940,594908,17032814,29.12,23.94
1,EDUCA18,52.50,0.00,0.00%,52.50,52.50,52.50,52.50,16,90,4720,58.36,46.39
2,FIBRAMQ12,43.33,-0.03,-0.07%,43.48,43.14,43.91,43.04,1135,1804951,78017102,45.27,27.73
3,FIBRAPL14,76.53,0.88,1.16%,75.77,76.04,76.62,74.91,3752,312725,23847457,83.97,64.05
4,FIBRAUP18,37.45,0.00,0.00%,37.45,37.45,37.45,37.45,3,16,599,41.00,17.27
5,FIHO12,7.59,0.03,0.40%,7.57,7.50,7.64,7.50,373,20531,155616,8.01,6.92
6,FINN13,4.75,0.00,0.00%,4.79,4.82,4.83,4.75,311,15725,75770,5.40,4.33
7,FMTY14,14.36,-0.06,-0.42%,14.33,14.36,14.49,14.17,11507,3702691,52938140,15.79,12.32
8,FNOVA17,41.69,-0.16,-0.38%,41.81,41.93,42.00,41.63,175,8866,371136,45.95,27.00
9,FPLUS16,5.05,-0.04,-0.79%,5.02,5.11,5.11,4.93,323,36701,184427,6.00,4.82


CSV analítico guardado en: D:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\20260825_232810_indice_fibras_amefibra.csv


### Exportar a archivo de Excel - xlsx (Ejecución opcional)

In [4]:
if EXPORTAR_CSV_EXCEL:
    ruta_csv_excel = exportar_csv_excel(df, RUTA_CSV_EXCEL)
    print(f"CSV compatible con Excel guardado en: {ruta_csv_excel}")

if EXPORTAR_XLSX:
    ruta_xlsx = exportar_xlsx(df, RUTA_XLSX)
    print(f"Excel guardado en: {ruta_xlsx}")

## Consulta de emisoras

In [5]:
try:
    df
except NameError:
    df = None

df_emisoras = mostrar_emisoras(df, CARPETA_SALIDA)

Fuente de emisoras: extracción de AMEFIBRA de esta corrida.
      Emisora
0    DANHOS13
1     EDUCA18
2   FIBRAMQ12
3   FIBRAPL14
4   FIBRAUP18
5      FIHO12
6      FINN13
7      FMTY14
8     FNOVA17
9     FPLUS16
10    FSHOP13
11     FUNO11
12     NEXT25
13     SOMA21
14  STORAGE18
CSV de emisoras guardado en: D:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\20260825_232810_list_of_tickers.csv


## Historial de distribuciones por FIBRA

### Fuentes evaluadas

| Fuente | Cobertura BMV | Datos de distribuciones | Acceso y límites |
|---|---|---|---|
| Relación con Inversionistas del emisor | Sí, por emisora | Fuente primaria; puede incluir fechas, importe y componentes fiscales en PDF/XLSX | Gratuita, sin API uniforme; requiere localizar y procesar reportes de cada emisor |
| AMEFIBRA | Sí, índice agregado | Cotización e indicadores del índice; no publica aquí un histórico normalizado de distribuciones | Consulta web pública; no se expone una API de dividendos en esta tabla |
| BMV/BIVA | Sí | Información oficial de emisoras y eventos, según disponibilidad del portal | Consulta pública, pero sin una API gratuita y estable para este flujo |
| FMP, Alpha Vantage, Twelve Data, EODHD, Nasdaq Data Link y Polygon | Cobertura mexicana variable | La cobertura y profundidad de dividendos para tickers BMV no está garantizada en el plan gratuito | Requieren revisar ticker, API key y límites por proveedor |
| `yfinance` | Sí para tickers Yahoo con sufijo `.MX`, cuando Yahoo dispone del evento | Fecha ex-dividendo y monto; no garantiza fecha de registro, pago ni componentes fiscales | Gratis y sin API key, pero es un cliente no oficial de Yahoo Finance y está sujeto a cambios y límites |

Se usa `yfinance` como respaldo reproducible porque las fuentes primarias no ofrecen una API homogénea. El resultado contiene la fecha ex-dividendo y el importe disponible en Yahoo; la fecha de registro, fecha de pago y componentes fiscales no se incluyen porque esta fuente no los entrega de forma confiable. El histórico se ordena del más antiguo al más reciente. `yield_pct` es el rendimiento de cada distribución respecto al cierre de su fecha ex-dividendo; `annualized_yield_pct` anualiza ese rendimiento usando `365 / días_del_periodo`. Para la primera fila se usa la mediana histórica de días entre distribuciones.

### Ticker a consultar

Elige, del desplegable, el ticker a consultar (mismo listado de la sección "Consula de emisoras"). Al correr esta celda se despliega el selector; cambia la selección y luego corre la celda de abajo para consultar el ticker elegido.

In [6]:
selector_ticker = seleccionar_ticker_interactivo(df_emisoras["Emisora"])

Dropdown(description='Ticker:', options=('DANHOS13', 'EDUCA18', 'FIBRAMQ12', 'FIBRAPL14', 'FIBRAUP18', 'FIHO12…

In [7]:
TICKER_SELECCIONADO = selector_ticker.value
historial_dividendos = probar_historial_dividendos(TICKER_SELECCIONADO, df_emisoras["Emisora"], CARPETA_SALIDA)

Ticker probado: DANHOS13. Registros: 48
Periodicidad detectada: trimestral
CSV generado: D:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\20260825_232813_DANHOS13_dividendos.csv


,ticker,ex_date,amount_mxn,close_on_ex_date_mxn,yield_pct,annualized_yield_pct,periodicity
38,DANHOS13,2024-05-09,0.450000,20.840000,2.159309,11.941633,trimestral
39,DANHOS13,2024-08-09,0.450000,20.410000,2.204802,8.747311,trimestral
40,DANHOS13,2024-11-12,0.450000,21.260000,2.116651,8.132396,trimestral
41,DANHOS13,2025-03-12,0.450000,21.879999,2.056673,6.255713,trimestral
42,DANHOS13,2025-05-12,0.450000,22.570000,1.993797,11.930097,trimestral
43,DANHOS13,2025-08-12,0.450000,25.790001,1.744862,6.922551,trimestral
44,DANHOS13,2025-11-12,0.450000,30.030001,1.498501,5.945142,trimestral
45,DANHOS13,2026-03-12,0.450000,25.059999,1.795690,5.461892,trimestral
46,DANHOS13,2026-05-12,0.450000,27.910000,1.612325,9.647520,trimestral
47,DANHOS13,2026-08-12,0.273086,28.530001,0.957189,3.797543,trimestral


## Ficha de rendimiento anual

La ficha usa el **año calendario** (`1 de enero` a `31 de diciembre`). Los pagos se filtran por `ex_date`, que es la fecha disponible en el historial de `yfinance`; no se inventa una fecha de pago que la fuente no proporciona. Los precios inicial y final son el primer y último cierre disponible dentro del año. El rendimiento por dividendos se calcula contra el precio inicial, y el rendimiento de capital contra la variación entre precio final e inicial. La ficha es informativa y no constituye una recomendación de inversión.

### Año a consultar

Elige, del desplegable, el año a consultar (solo se muestran los años con distribuciones disponibles para el ticker). Al correr esta celda se despliega el selector; cambia la selección y luego corre la celda de abajo para generar la ficha con el año elegido.

In [8]:
AÑOS_DISPONIBLES = obtener_anios_disponibles(historial_dividendos)
selector_anio = seleccionar_anio_interactivo(AÑOS_DISPONIBLES)

Hay información disponible de 2014 a 2026.


Dropdown(description='Año:', options=(2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018, 2017, 2016, 2015, …

In [9]:
AÑO_SELECCIONADO = selector_anio.value
ruta_ficha = mostrar_ficha_rendimiento(TICKER_SELECCIONADO, AÑO_SELECCIONADO, CARPETA_SALIDA, historial_dividendos)

Ficha generada: D:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\20260825_232814_DANHOS13_2026_ficha_rendimiento.html


ex_date,amount_mxn,yield_pct
2026-03-12,$0.4500,1.80%
2026-05-12,$0.4500,1.61%
2026-08-12,$0.2731,0.96%


In [10]:
ruta_pdf_rendimiento = exportar_ficha_a_pdf(
    ruta_ficha, TICKER_SELECCIONADO, "rendimiento anual", AÑO_SELECCIONADO, CARPETA_FICHAS_PDF
)
print(f"PDF generado: {ruta_pdf_rendimiento}")

PDF generado: D:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\fichas\2026-08-25_2328_DANHOS13_rendimiento-anual_2026.pdf


## Ficha de ejemplo para cliente (demostración)

Versión de demostración de la ficha, pensada para mostrarle al cliente el aspecto del entregable final (escenario de inversión, distribuciones mensuales y rendimiento total en el año), con un diseño distinto al de la ficha de rendimiento anterior. Usa el mismo ticker y año ya elegidos arriba y los mismos datos reales (`historial_dividendos`); no inventa cifras. Es informativa y no constituye una recomendación de inversión.

In [11]:
ruta_ficha_ejemplo = mostrar_ficha_ejemplo_cliente(TICKER_SELECCIONADO, AÑO_SELECCIONADO, CARPETA_SALIDA, historial_dividendos)

Ficha de ejemplo generada: D:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\20260825_232818_DANHOS13_2026_ficha_ejemplo_cliente.html


In [12]:
ruta_pdf_ejemplo = exportar_ficha_a_pdf(
    ruta_ficha_ejemplo, TICKER_SELECCIONADO, "ficha ejemplo cliente", AÑO_SELECCIONADO, CARPETA_FICHAS_PDF
)
print(f"PDF generado: {ruta_pdf_ejemplo}")

PDF generado: D:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\fichas\2026-08-25_2328_DANHOS13_ficha-ejemplo-cliente_2026.pdf
